In [ ]:
class ParserFNC:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def match(self, token):
        if self.pos < len(self.tokens) and self.tokens[self.pos] == token:
            self.pos += 1
            return True
        return False

    def S0(self):
        pos_backup = self.pos
        if self.T() and self.E_prime(): return True
        self.pos = pos_backup
        if self.F() and self.T_prime(): return True
        self.pos = pos_backup
        if self.L() and self.X1(): return True
        self.pos = pos_backup
        if self.match("id"): return True
        self.pos = pos_backup
        return False

    def E(self):
        pos_backup = self.pos
        if self.T() and self.E_prime(): return True
        self.pos = pos_backup
        if self.F() and self.T_prime(): return True
        self.pos = pos_backup
        if self.L() and self.X1(): return True
        self.pos = pos_backup
        if self.match("id"): return True
        self.pos = pos_backup
        return False

    def E_prime(self):
        pos_backup = self.pos
        if self.P() and self.X2(): return True
        self.pos = pos_backup
        if self.P() and self.T(): return True
        self.pos = pos_backup
        return False

    def T(self):
        pos_backup = self.pos
        if self.F() and self.T_prime(): return True
        self.pos = pos_backup
        if self.L() and self.X1(): return True
        self.pos = pos_backup
        if self.match("id"): return True
        self.pos = pos_backup
        return False

    def T_prime(self):
        pos_backup = self.pos
        if self.M() and self.X3(): return True
        self.pos = pos_backup
        if self.M() and self.F(): return True
        self.pos = pos_backup
        return False

    def F(self):
        pos_backup = self.pos
        if self.L() and self.X1(): return True
        self.pos = pos_backup
        if self.match("id"): return True
        self.pos = pos_backup
        return False

    def X1(self):
        pos_backup = self.pos
        if self.E() and self.R(): return True
        self.pos = pos_backup
        return False

    def X2(self):
        pos_backup = self.pos
        if self.T() and self.E_prime(): return True
        self.pos = pos_backup
        return False

    def X3(self):
        pos_backup = self.pos
        if self.F() and self.T_prime(): return True
        self.pos = pos_backup
        return False

    def P(self): return self.match("+")
    def M(self): return self.match("*")
    def L(self): return self.match("(")
    def R(self): return self.match(")")

def analyser(chaine):
    """Détermine si la chaine est générée par la grammaire en FNC."""
    parser = ParserFNC(chaine)
    return parser.S0() and parser.pos == len(chaine)

# ==========================================
# TESTS QUESTION 2.2
# ==========================================
print("--- Tests Analyseur Booléen ---")
print(analyser(["id", "+", "id"]))               # True
print(analyser(["id", "*", "id"]))               # True
print(analyser(["(", "id", "+", "id", ")"]))     # True
print(analyser(["id", "+"]))                     # False
print(analyser(["id", "+", "id", "*", "id"]))    # True

: 

In [ ]:
# Extension optionnelle : Modifiez votre analyseur pour qu’il retourne un arbre de dérivation (AST - Abstract Syntax Tree).

class ASTNode:
    def __init__(self, type_, value=None, children=None):
        self.type = type_
        self.value = value
        self.children = children or []

class ParserFNC_AST:
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0
        self.id_counter = 0

    def get_state(self):
        return self.pos, self.id_counter

    def restore_state(self, state):
        self.pos, self.id_counter = state

    def match(self, token):
        if self.pos < len(self.tokens) and self.tokens[self.pos] == token:
            self.pos += 1
            return True
        return False

    def S0(self):
        state = self.get_state()
        t = self.T()
        if t:
            ep = self.E_prime()
            if ep: return ASTNode('op', ep[0], [t, ep[1]])
        self.restore_state(state)
        
        f = self.F()
        if f:
            tp = self.T_prime()
            if tp: return ASTNode('op', tp[0], [f, tp[1]])
        self.restore_state(state)
        
        l = self.L()
        if l:
            x1 = self.X1()
            if x1: return x1
        self.restore_state(state)
        
        return self.match_id()

    def E(self):
        state = self.get_state()
        t = self.T()
        if t:
            ep = self.E_prime()
            if ep: return ASTNode('op', ep[0], [t, ep[1]])
        self.restore_state(state)
        
        f = self.F()
        if f:
            tp = self.T_prime()
            if tp: return ASTNode('op', tp[0], [f, tp[1]])
        self.restore_state(state)
        
        l = self.L()
        if l:
            x1 = self.X1()
            if x1: return x1
        self.restore_state(state)
        
        return self.match_id()

    def E_prime(self):
        state = self.get_state()
        p = self.P()
        if p:
            x2 = self.X2()
            if x2: return [p, x2]
        self.restore_state(state)
        
        p = self.P()
        if p:
            t = self.T()
            if t: return [p, t]
        self.restore_state(state)
        return None

    def T(self):
        state = self.get_state()
        f = self.F()
        if f:
            tp = self.T_prime()
            if tp: return ASTNode('op', tp[0], [f, tp[1]])
        self.restore_state(state)
        
        l = self.L()
        if l:
            x1 = self.X1()
            if x1: return x1
        self.restore_state(state)
        
        return self.match_id()

    def T_prime(self):
        state = self.get_state()
        m = self.M()
        if m:
            x3 = self.X3()
            if x3: return [m, x3]
        self.restore_state(state)
        
        m = self.M()
        if m:
            f = self.F()
            if f: return [m, f]
        self.restore_state(state)
        return None

    def F(self):
        state = self.get_state()
        l = self.L()
        if l:
            x1 = self.X1()
            if x1: return x1
        self.restore_state(state)
        return self.match_id()

    def X1(self):
        state = self.get_state()
        e = self.E()
        if e:
            r = self.R()
            if r: return e 
        self.restore_state(state)
        return None

    def X2(self):
        state = self.get_state()
        t = self.T()
        if t:
            ep = self.E_prime()
            if ep: return ASTNode('op', ep[0], [t, ep[1]])
        self.restore_state(state)
        return None

    def X3(self):
        state = self.get_state()
        f = self.F()
        if f:
            tp = self.T_prime()
            if tp: return ASTNode('op', tp[0], [f, tp[1]])
        self.restore_state(state)
        return None

    def P(self): return "+" if self.match("+") else None
    def M(self): return "*" if self.match("*") else None
    def L(self): return "(" if self.match("(") else None
    def R(self): return ")" if self.match(")") else None

    def match_id(self):
        if self.match("id"):
            name = chr(120 + self.id_counter) # Génère x, y, z...
            self.id_counter += 1
            return ASTNode('id', name)
        return None


def analyser_ast(tokens):
    """Retourne l'AST si la chaîne est valide, None sinon."""
    parser = ParserFNC_AST(tokens)
    ast = parser.S0()
    if ast and parser.pos == len(tokens):
        return ast
    return None

def precedence(op):
    if op == '*': return 2
    if op == '+': return 1
    return 0

def generer_code(node):
    """Génère le code Python évaluable à partir de l'AST."""
    if node is None:
        return ""
        
    if node.type == 'id':
        return node.value

    elif node.type == 'op':
        left = generer_code(node.children[0])
        right = generer_code(node.children[1])
        op = node.value

        # Gestion des parenthèses selon la priorité
        if node.children[0].type == 'op':
            if precedence(node.children[0].value) < precedence(op):
                left = f"({left})"
        if node.children[1].type == 'op':
            if precedence(node.children[1].value) < precedence(op):
                right = f"({right})"

        return f"{left} {op} {right}"


# ==========================================
# TESTS QUESTION 2.3
# ==========================================
print("\n--- Tests Génération de Code (AST) ---")

tokens1 = ["id", "+", "id"]
ast1 = analyser_ast(tokens1)
print(f"{tokens1} ->", generer_code(ast1))  # x + y

tokens2 = ["(", "id", "+", "id", ")", "*", "id"]
ast2 = analyser_ast(tokens2)
print(f"{tokens2} ->", generer_code(ast2))  # (x + y) * z